# Between: synthetic ML walkthrough

This notebook demonstrates the pipeline described in `COMPLETE PROJECT.pdf`. All rows and outcome totals are simulated. Results do not establish diagnostic accuracy, causation or future illness. See `docs/machine-learning.md` for assumptions and the provisional three-class target. Running all cells regenerates local synthetic artifacts; it does not access the app database.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "ml":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from ml.generate_data import generate_data
from ml.train import clean_data, train
from ml.schema import FEATURES, TARGET
print(TARGET)


## Generate and inspect fictional data

The raw data includes duplicate IDs, invalid rows and missing values to demonstrate actual preprocessing. Background attributes are simulated for inspection and excluded from model inputs.


In [ ]:
raw = generate_data(samples=6000, seed=2026)
raw_path = ROOT / "ml/data/synthetic_raw.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw.to_csv(raw_path, index=False)
raw.head()


In [ ]:
cleaned, cleaning_report = clean_data(raw)
print(cleaning_report)
cleaned["risk_class"].value_counts()


## Features and leakage prevention

Only age and the six versioned behaviour fields enter the model. Simulated PHQ-9/GAD-7 totals are used only for the provisional outcome label. Imputation, scaling and encoding are fitted within training folds. The command below compares Logistic Regression, Decision Tree and Random Forest using training CV and validation macro-F1, refits the selected configuration on train+validation and evaluates it on the test partition.


In [ ]:
print("Model inputs:", list(FEATURES))
report = train(raw_path, ROOT / "ml/artifacts", seed=2026)
pd.DataFrame([{
    "algorithm": item["algorithm"],
    "CV macro F1": item["cv_f1_macro"],
    "validation macro F1": item["validation"]["f1_macro"],
    "validation AUC": item["validation"]["roc_auc_ovr_macro"],
} for item in report["comparisons"]])


In [ ]:
print("Selected:", report["algorithm"])
print("Final synthetic test:", report["metrics"])
pd.DataFrame(report["confusion_matrix"], index=report["label_order"], columns=report["label_order"])


## Export research figures and try inference

Probabilities below are uncalibrated simulation class outputs. Missing or invalid inference fields receive no estimate; psychometric scores are never created from habit answers.


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "tools/plot_ml_evaluation.py")], cwd=ROOT, check=True)
from IPython.display import Image, display
display(Image(filename=str(ROOT / "ml/artifacts/figures/confusion_matrix.png")))


In [ ]:
from ml.service import ModelService
model = ModelService(ROOT / "ml/artifacts")
example = {"age":19,"daily_time":"1to3","passive":"often","active":"rarely",
           "night":"sometimes","cyberbullying":"never","regular_count":"2"}
model.predict(example)


## Interpretation

This is a development demonstration trained and evaluated on invented relationships. The target definition is provisional. No real participant study, external validation, questionnaire administration, probability calibration or clinical release has occurred. Restart Flask after regenerating artifacts. Existing saved assessments keep their original prediction snapshots.
